# Ejecución completa del Tema 55 (Alejandro Soto Antezana, 2024200531K)
Ejecute las celdas **en orden**, de arriba hacia abajo, con el botón ▶ de cada una.

## Celda 1. Conectar Google Drive y entrar a la carpeta del proyecto

In [1]:
from google.colab import drive
drive.mount('/content/drive')

CARPETA = '/content/drive/MyDrive/3_Base_de_datos_y_codigo_Alejandro_Soto'
%cd "$CARPETA"
!ls

Mounted at /content/drive
/content/drive/MyDrive/3_Base_de_datos_y_codigo_Alejandro_Soto
codigo		  diccionario_variables.md  log_ejecucion.txt  salidas
datos_crudos	  ejecutar_en_colab.ipynb   README.md
datos_procesados  incidencias_fuente.md     requirements.txt


## Celda 2. Instalar librerías

In [2]:
!pip install -q -r requirements.txt

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


## Celda 3. Crear el archivo .env (solo en Colab; NUNCA se sube a GitHub)

In [ ]:
%%writefile .env
SEC_USER_AGENT="Alejandro Soto Antezana e_2024200531K@uncp.edu.pe"

## Celda 4. Vía 1: API (BCRP + Yahoo Finance)

In [ ]:
!python codigo/01_extraccion_api.py

## Celda 5. Vía 2: descarga programática (SEC + FRED)

In [ ]:
!python codigo/02_scraping_web.py

## Celda 6. Limpieza, unión, hash y tabla de cotejo

In [ ]:
!python codigo/03_limpieza_datos.py

## Celda 7. Análisis y Monte Carlo (tablas y figuras en salidas/)

In [3]:
!python codigo/04_analisis.py

2026-11-27 11:44:15 (Lima) | 04_analisis | Base: 5221 días, de 2004-11-18 a 2026-08-31; demanda medida con 'gld_monto_musd'
2026-11-27 11:44:22 (Lima) | 04_analisis | Validación: VAN modelo 205.3 vs oficial 191.0 US$ mm
2026-11-27 11:44:23 (Lima) | 04_analisis | Monte Carlo: 10000 simulaciones por escenario, semilla 531, bloques de 21 días, P0 mercado = 4,437.38 US$/oz, TC0 = 3.3629
2026-11-27 11:44:53 (Lima) | 04_analisis | Tablas: 12 | Figuras: 11 (PNG 300 dpi + PDF) | Cointegración oro-demanda: False

=== CONTRASTE DE HIPÓTESIS ===
                                                                      Hipótesis  Estadístico       Valor p        Decisión
            HG: precio, demanda, TC y riesgo país explican conjuntamente el VAN  5290.890331  0.000000e+00   Se rechaza H0
               HE1: el precio del oro es el factor de mayor efecto sobre el VAN   135.132960  0.000000e+00    Se acepta H1
      HE2: la demanda de inversión en oro se asocia positivamente con su precio     0.0105

## Celda 7b. Figuras 3D (8 a 12) y tabla 11 de evidencia de la SEC
Ejecútela **después** de la celda 7, porque usa `salidas/simulaciones_van.csv` y `salidas/tabla7_parametros_y_validacion.csv`.

In [4]:
!python codigo/05_figuras_3d.py

Guardada: figura8_superficie_van_precio_tasa
Guardada: figura9_superficie_van_precio_costo
Guardada: figura10_histograma3d_tasa_van
Guardada: figura11_cordillera_escenarios
Guardada: figura12_nube3d_rendimientos
Pendiente TC sobre oro (piso de la figura 12): -0.0369; días graficados: 5161
Guardada: tabla11_evidencia_sec.tex


## Celda 8. requirements.txt con versiones exactas y README rellenado por código
Revise que `ENLACE_REPO` sea igual al enlace de la barra del navegador de su repositorio.
Esta celda **reescribe** las líneas del README cada vez que se ejecuta (sirve aunque ya se haya ejecutado antes).

In [5]:
import sys, re, hashlib, pandas as pd
from importlib.metadata import version
ENLACE_REPO = "https://github.com/Alejandro-econ/tema55-montecarlo-san-gabriel"   # <- revise que sea el suyo

# 1) requirements.txt con las versiones exactas instaladas
libs = ["requests", "pandas", "numpy", "yfinance", "pdfplumber", "statsmodels", "scipy",
        "matplotlib", "openpyxl", "python-dotenv"]
lineas = [f"{l}=={version(l)}" for l in libs]
open("requirements.txt", "w").write("\n".join(lineas) + "\n")

# 2) Datos reales de la base procesada (solo se lee; no se modifica)
proc = "datos_procesados/datos_procesados_2024200531K.csv"
base = pd.read_csv(proc)
sha = hashlib.sha256(open(proc, "rb").read()).hexdigest()

# 3) Reescribir cada línea del README (funciona con marcadores {{...}} o con valores viejos)
nuevas = {
    r"^\| Repositorio \|.*$": f"| Repositorio | {ENLACE_REPO} |",
    r"^- \*\*Primera y última fecha.*$": f"- **Primera y última fecha con datos reales en la base procesada:** "
        f"{base['fecha'].iloc[0]} a {base['fecha'].iloc[-1]} ({len(base):,} días).",
    r"^- Python:.*$": f"- Python: {sys.version.split()[0]}",
    r"^- Librerías:.*$": "- Librerías: ver `requirements.txt`, generado con las versiones exactas usadas: " + ", ".join(lineas),
    r"^- \*\*SHA-256 de .*$": f"- **SHA-256 de `datos_procesados/datos_procesados_2024200531K.csv`:** `{sha}`",
}
readme = open("README.md", encoding="utf-8").read()
for patron, linea in nuevas.items():
    readme, n = re.subn(patron, lambda m: linea, readme, flags=re.M)
    print(("OK   " if n == 1 else "FALTA") + "  " + linea[:90])
open("README.md", "w", encoding="utf-8").write(readme)
print("\nQuedan marcadores {{...}}:", "SÍ, avise a Claude" if "{{" in readme else "no")
print("\nrequirements.txt:\n" + open("requirements.txt").read())

OK     | Repositorio | https://github.com/Alejandro-econ/tema55-montecarlo-san-gabriel |
OK     - **Primera y última fecha con datos reales en la base procesada:** 2004-11-18 a 2026-08-3
OK     - Python: 3.13.18
OK     - Librerías: ver `requirements.txt`, generado con las versiones exactas usadas: requests==
OK     - **SHA-256 de `datos_procesados/datos_procesados_2024200531K.csv`:** `5f76f03e7199fb95181

Quedan marcadores {{...}}: no

requirements.txt:
requests==2.32.4
pandas==2.2.3
numpy==2.1.3
yfinance==0.2.66
pdfplumber==0.11.10
statsmodels==0.15.0
scipy==1.16.3
matplotlib==3.10.0
openpyxl==3.1.5
python-dotenv==1.2.3



## Celda 9. Empaquetar lo que debo enviar a Claude para revisar y redactar el artículo

In [ ]:
import shutil, os
os.makedirs("/content/para_claude", exist_ok=True)
for f in ["log_ejecucion.txt", "README.md", "datos_crudos/datos_crudos_2024200531K_sec_trs_parametros.csv",
          "datos_procesados/parametros_proyecto_2024200531K.csv"]:
    if os.path.exists(f): shutil.copy(f, "/content/para_claude/")
shutil.copytree("salidas", "/content/para_claude/salidas", dirs_exist_ok=True)
shutil.make_archive("/content/resultados_para_claude", "zip", "/content/para_claude")
from google.colab import files
files.download("/content/resultados_para_claude.zip")